<a href="https://colab.research.google.com/github/karthik-srivathsa-05/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthik-srivathsa-05/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one daily page observation for one client.**

The warehouse fact table records observations at the `(client_hash_id, content_hash_id, report_date)` level. The same client-page therefore appears across multiple dates.

For this lane, I use February 2026 as the feature window and February 28, 2026 as the decision cutoff.

- **Feature window:** February 2026
- **Decision cutoff:** 2026-02-28
- **Future outcome window:** March 2026
- **Main table:** `fact_content_daily_performance`

The model features will be aggregated to one row per client-page using only information available through February 28.

The target will be based on the following month's observed performance, so March information is treated as future information and is not used as a feature.

I deliberately exclude June 2026 from development because it is the final month in the warehouse panel and should remain a sealed test period.

The output is intended to support a ranked content-review queue. It is decision-support, not a causal estimate of whether refreshing a page will improve its performance.

In [8]:
import os
import getpass
import duckdb
import pandas as pd

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Enter Hugging Face READ token: ")

HF_TOKEN = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    "CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet'"
    f")"
)

FACT_MAR = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Decision cutoff: 2026-02-28")
print("Future outcome window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Decision cutoff: 2026-02-28
Future outcome window: March 2026


## 2. Fields: feature / label / context / excluded

### Features

I will use at most five features, all calculated from the February 2026 feature window:

1. **impressions_30d** — total GSC impressions during February, representing recent search visibility.
2. **clicks_30d** — total GSC clicks during February, representing recent search traffic.
3. **avg_position** — average observed GSC search position during February.
4. **days_with_impressions** — number of February days on which the page received at least one GSC impression.
5. **organic_sessions_30d** — total organic sessions during February from the available GA4 data.

These features are calculated only from February observations, so they are available at the February 28 decision cutoff.

### Label / proxy

The label is whether the page experiences a meaningful decline in March compared with its February baseline.

I define the February baseline and March outcome using average daily GSC impressions so that the comparison is not distorted by February having 28 days and March having 31 days.

A page is labelled `1` when its March average daily impressions are at least 20% lower than its February average daily impressions. Otherwise it is labelled `0`.

This is an observed directional outcome, not a causal measure of whether a content refresh caused or prevented the decline.

### Context

- `client_hash_id` identifies the client and is used for grouping and client-aware validation.
- `content_hash_id` identifies the page and is used for joins and review-queue output.
- `report_date` defines the time window.
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available` describe data availability.

The identifiers are retained for grouping and traceability but are not used as model features.

### Excluded

I deliberately exclude all March performance variables from the feature set because March occurs after the February decision cutoff.

I also exclude June 2026 from development because it is the final month of the warehouse panel and should remain a sealed test period.

I exclude client and content identifiers from the model because they identify entities rather than provide meaningful behavioral signals.

## 3. Verify it with queries

### Query 1 — Grain

The intended warehouse grain is one row per `(client_hash_id, content_hash_id, report_date)`.

This query checks for duplicate observations at that level. A result of zero means there are no duplicate client-page-date observations in the February slice.

In [9]:
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {FACT_FEB}
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 20
""").df()

print("Duplicate client-content-date observations:", len(grain_check))
display(grain_check)

Duplicate client-content-date observations: 0


,client_hash_id,content_hash_id,report_date,row_count


### Query 2 — February slice size and date span

This query checks that the selected February partition contains the expected reporting period and records the number of observations used in the feature window.

In [10]:
slice_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {FACT_FEB}
""").df()

display(slice_check)

,row_count,min_report_date,max_report_date
0,7355108,2026-02-01,2026-02-28


The February slice contains **7,355,108 daily observations**, covering **2026-02-01 through 2026-02-28**.

The presence of multiple rows for the same client-page is expected because the warehouse grain is daily rather than one row per page for the whole month.

### Query 3 — Data availability

For this lane, GSC availability is required because the features and label use GSC search performance.

I therefore check the February rows using `gsc_data_available IS TRUE` rather than relying on a truthy value or assuming that every row contains usable GSC data.

In [11]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS rows_with_gsc_available,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS rows_without_gsc_available
FROM {FACT_FEB}
""").df()

display(availability_check)

,total_rows,rows_with_gsc_available,rows_without_gsc_available
0,7355108,2621783,4733325


## 4. Five-feature frame

I aggregate the daily February observations into one row per client-page.

The five features are:

| Feature | Why it is available at the decision moment |
|---|---|
| `impressions_30d` | It is calculated from GSC impressions observed during February, ending before the decision. |
| `clicks_30d` | It is calculated from GSC clicks observed during February, ending before the decision. |
| `avg_position` | It is calculated from February GSC position observations available before the decision. |
| `days_with_impressions` | It counts February days with at least one observed impression, so it uses only historical observations. |
| `organic_sessions_30d` | It is calculated from February organic sessions available from GA4. |

No March performance variable is included in this feature frame.

In [12]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(COALESCE(gsc_impressions, 0)) AS impressions_30d,

    SUM(COALESCE(gsc_clicks, 0)) AS clicks_30d,

    AVG(gsc_avg_position) AS avg_position,

    COUNT(*) FILTER (
        WHERE COALESCE(gsc_impressions, 0) > 0
    ) AS days_with_impressions,

    SUM(COALESCE(sessions_organic, 0)) AS organic_sessions_30d

FROM {FACT_FEB}

WHERE
    gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Feature rows:", len(feature_frame))
print("Feature columns:", list(feature_frame.columns))

display(feature_frame.head())

Feature rows: 153559
Feature columns: ['client_hash_id', 'content_hash_id', 'impressions_30d', 'clicks_30d', 'avg_position', 'days_with_impressions', 'organic_sessions_30d']


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position,days_with_impressions,organic_sessions_30d
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,5.500000,2,0.0
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,5.000000,4,0.0
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,5.262333,26,0.0
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,6.407819,28,0.0
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,6.961538,13,0.0


## 5. Future label

The February feature frame represents information available at the decision cutoff.

The March label is constructed separately from March observations.

I compare average daily impressions rather than monthly totals because February has 28 days and March has 31 days.

A page is labelled as declining when March average daily impressions are at least 20% below its February average daily impressions.

This label is used only as the future outcome for training/evaluation. It must not be included in the February feature frame.

In [13]:
feb_baseline = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(COALESCE(gsc_impressions, 0)) AS feb_avg_daily_impressions
FROM {FACT_FEB}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

mar_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(COALESCE(gsc_impressions, 0)) AS mar_avg_daily_impressions
FROM {FACT_MAR}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

label_frame = feb_baseline.merge(
    mar_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

label_frame["impression_change_pct"] = (
    (
        label_frame["mar_avg_daily_impressions"]
        - label_frame["feb_avg_daily_impressions"]
    )
    / label_frame["feb_avg_daily_impressions"].replace(0, pd.NA)
)

label_frame["is_declining"] = (
    label_frame["impression_change_pct"] <= -0.20
).astype("int8")

print("Label rows:", len(label_frame))
print("\nLabel distribution:")
print(label_frame["is_declining"].value_counts().sort_index())

display(label_frame.head())

Label rows: 134238

Label distribution:
is_declining
0    100575
1     33663
Name: count, dtype: int64


,client_hash_id,content_hash_id,feb_avg_daily_impressions,mar_avg_daily_impressions,impression_change_pct,is_declining
0,client_e547b89c05043229,content_1eea820697c3b95a,10.678571,10.862069,0.017184,0
1,client_e547b89c05043229,content_9abd8b303f805847,26.178571,501.241379,18.147010,0
2,client_e547b89c05043229,content_5f58c55cbfee172a,18.357143,13.344828,-0.273044,1
3,client_e547b89c05043229,content_6fe390ba3af1e456,104.678571,161.965517,0.547265,0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,34.642857,34.620690,-0.000640,0


## 6. Feature + label frame

The final development frame joins the February features to the future March outcome using the client-page identifiers.

The identifiers are retained for traceability and client-aware validation, but they are not treated as predictive features.

In [14]:
model_frame = feature_frame.merge(
    label_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining",
            "impression_change_pct",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model rows:", len(model_frame))
print("Model columns:", len(model_frame.columns))

display(model_frame.head())

Model rows: 134238
Model columns: 9


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position,days_with_impressions,organic_sessions_30d,is_declining,impression_change_pct
0,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,5.000000,4,0.0,0,0.200000
1,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,5.262333,26,0.0,0,-0.168155
2,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,6.407819,28,0.0,0,-0.019904
3,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,6.961538,13,0.0,0,0.083333
4,client_3ffa76342f366962,content_0674cc4ae0f68a90,74.0,1.0,8.063910,19,0.0,1,-0.529279


## 7. The trap — deliberate label leakage

To demonstrate leakage, I intentionally add `impression_change_pct` as a feature.

This value uses March performance, which is only known after the February decision moment. Therefore it would not be available when the review queue is created.

Because the leaked feature is directly derived from the future outcome, a quick model can appear unrealistically strong.

The purpose of this experiment is to show why future outcome information must be removed from the honest feature set.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

leaky_features = [
    "impressions_30d",
    "clicks_30d",
    "avg_position",
    "days_with_impressions",
    "organic_sessions_30d",
    "impression_change_pct",  # DELIBERATE LEAK
]

leaky_df = model_frame.dropna(
    subset=leaky_features + ["is_declining"]
).copy()

X = leaky_df[leaky_features]
y = leaky_df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train, y_train)

leaky_scores = leaky_model.predict_proba(X_test)[:, 1]

print(
    "Leaky ROC-AUC:",
    round(roc_auc_score(y_test, leaky_scores), 3)
)

Leaky ROC-AUC: 1.0


## 8. Remove the leakage

The `impression_change_pct` feature is deleted because it contains March information.

The honest feature set contains only February observations that would have been available at the February 28 decision cutoff.

The difference between the leaky and honest results is the practical leakage lesson: a strong score is not useful if the model could not have known the feature when the decision was made.

In [16]:
honest_features = [
    "impressions_30d",
    "clicks_30d",
    "avg_position",
    "days_with_impressions",
    "organic_sessions_30d",
]

honest_df = model_frame.dropna(
    subset=honest_features + ["is_declining"]
).copy()

X = honest_df[honest_features]
y = honest_df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_scores = honest_model.predict_proba(X_test)[:, 1]

print(
    "Honest ROC-AUC:",
    round(roc_auc_score(y_test, honest_scores), 3)
)

Honest ROC-AUC: 0.681


## 9. Leakage comparison

The comparison is intentionally simple.

The leaky experiment is expected to produce an artificially strong score because it gives the model direct information about the future outcome.

The honest experiment removes that future information and therefore provides a more realistic estimate of what can be learned from information available at the decision moment.

The honest score is the number that should be retained for further modeling work. The leaky result is shown only as a warning and is not used for model selection.

In [17]:
# Recalculate the leaky experiment's test split so both scores
# are paired with their correct test labels.

X_leak = leaky_df[leaky_features]
y_leak = leaky_df["is_declining"]

X_leak_train, X_leak_test, y_leak_train, y_leak_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.25,
    random_state=42,
    stratify=y_leak
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_leak_train, y_leak_train)

leaky_scores = leaky_model.predict_proba(X_leak_test)[:, 1]

leaky_auc = roc_auc_score(
    y_leak_test,
    leaky_scores
)

honest_auc = roc_auc_score(
    y_test,
    honest_scores
)

comparison = pd.DataFrame({
    "experiment": [
        "Deliberately leaky",
        "Honest"
    ],
    "roc_auc": [
        leaky_auc,
        honest_auc
    ]
})

comparison["roc_auc"] = comparison["roc_auc"].round(3)

display(comparison)

,experiment,roc_auc
0,Deliberately leaky,1.000
1,Honest,0.681


## 10. Data limits

One limitation of this slice is that it contains daily observations rather than a single page-level observation for the entire month. The same client-page therefore appears across many dates, so the feature frame must aggregate the daily rows before modeling.

Another limitation is that GSC and GA4 availability is not uniform across all rows. Pages or clients without the required data may be excluded from the corresponding feature or label calculation, which can make the modeling population different from the full warehouse population.

The February-to-March setup also gives only one future outcome month for this development example. It is therefore not enough by itself to establish that the learned relationship will generalize to other months.

Finally, the observed decline label describes a subsequent performance change. It does not establish that updating a page would cause its performance to improve.

## Final contract summary

| Contract item | Decision |
|---|---|
| **Unit of analysis** | One daily page observation for one client in the warehouse; aggregated to one client-page for modeling |
| **Feature window** | February 2026 |
| **Decision cutoff** | 2026-02-28 |
| **Future outcome window** | March 2026 |
| **Task** | Predict/rank pages based on subsequent performance decline |
| **Label** | March average daily impressions at least 20% below February average daily impressions |
| **Features** | February impressions, clicks, average position, days with impressions, organic sessions |
| **Context** | Client/page identifiers, report date, data availability flags |
| **Excluded** | March performance variables, June 2026 development data, identifiers as model features |
| **Main limitation** | Only one February→March outcome window is used in this development example, and data availability is not uniform |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.